## Import

In [1]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
import pyarrow
from sqlalchemy import create_engine
from dotenv import load_dotenv

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

## Configuration

In [2]:
load_dotenv()

USER = os.getenv("LOCAL_DB_USER")
PASSWORD = os.getenv("LOCAL_DB_PASS")
HOST = os.getenv("LOCAL_DB_HOST")
DBNAME = os.getenv("LOCAL_DB_NAME")
PORT = os.getenv("LOCAL_DB_PORT")

DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"
engine = create_engine(DATABASE_URL)

## Load dataset

In [3]:
with engine.connect() as conn:
    df_category = pl.read_database("SELECT * FROM category", conn)
    df_seller = pl.read_database("SELECT * FROM seller", conn)
    df_product = pl.read_database("SELECT * FROM product", conn)
    df_price_offer = pl.read_database("SELECT * FROM price_offer", conn)
    df_review = pl.read_database("SELECT * FROM review", conn)
    df_reviewer = pl.read_database("SELECT * FROM reviewer", conn)
    df_service = pl.read_database("SELECT * FROM service", conn)
    df_coupon = pl.read_database("SELECT * FROM coupon", conn)
    df_offer_coupon = pl.read_database("SELECT * FROM offer_coupon", conn)
    df_offer_service = pl.read_database("SELECT * FROM offer_service", conn)

## Bảng Category

In [4]:
print(f"Kích thước bảng Category: {df_category.shape}")
print(f"Kiểu dữ liệu của bảng Category:\n{df_category.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Category:")
display(df_category.null_count())

display(df_category.head())

Kích thước bảng Category: (3901, 7)
Kiểu dữ liệu của bảng Category:
[String, String, String, Int64, String, String, Boolean]
Kiểm tra số lượng dữ liệu null trong bảng Category:


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
u32,u32,u32,u32,u32,u32,u32
0,0,26,0,0,0,0


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
str,str,str,i64,str,str,bool
"""1789""","""Điện Thoại - Máy Tính Bảng""",null,1,"""Điện Thoại - Máy Tính Bảng""","""https://tiki.vn/dien-thoai-may…",true
"""1686""","""Giày - Dép nam""",null,1,"""Giày - Dép nam""","""https://tiki.vn/giay-dep-nam/c…",true
"""8322""","""Nhà Sách Tiki""",null,1,"""Nhà Sách Tiki""","""https://tiki.vn/nha-sach-tiki/…",true
"""4221""","""Điện Tử - Điện Lạnh""",null,1,"""Điện Tử - Điện Lạnh""","""https://tiki.vn/dien-tu-dien-l…",true
"""8371""","""Đồng hồ và Trang sức""",null,1,"""Đồng hồ và Trang sức""","""https://tiki.vn/dong-ho-va-tra…",true


In [5]:
df_category.unique(subset=["category_id"], keep='first') \
            .drop('is_scanned') \
            .with_columns([
                pl.col("category_id").cast(pl.Int64),
                pl.col("parent_category_id").cast(pl.Int64),
                pl.col("level").cast(pl.Int8),
                pl.col("category_path").str.strip_chars(),
                pl.col("category_name").str.strip_chars(),
                pl.col("category_url").str.strip_chars()
            ]) \
            .with_columns([
                pl.col('parent_category_id').fill_null(0)
            ]) \
            .sort(by=['level', 'category_id'])

category_id,category_name,parent_category_id,level,category_path,category_url
i64,str,i64,i8,str,str
915,"""Thời trang nam""",0,1,"""Thời trang nam""","""https://tiki.vn/thoi-trang-nam…"
931,"""Thời trang nữ""",0,1,"""Thời trang nữ""","""https://tiki.vn/thoi-trang-nu/…"
976,"""Túi thời trang nữ""",0,1,"""Túi thời trang nữ""","""https://tiki.vn/tui-vi-nu/c976"""
1520,"""Làm Đẹp - Sức Khỏe""",0,1,"""Làm Đẹp - Sức Khỏe""","""https://tiki.vn/lam-dep-suc-kh…"
1686,"""Giày - Dép nam""",0,1,"""Giày - Dép nam""","""https://tiki.vn/giay-dep-nam/c…"
…,…,…,…,…,…
67877,"""Luyện thi N1""",67872,6,"""Nhà Sách Tiki > Sách tiếng Việ…","""https://tiki.vn/luyen-thi-n1/c…"
67878,"""Luyện thi N2""",67872,6,"""Nhà Sách Tiki > Sách tiếng Việ…","""https://tiki.vn/luyen-thi-n2/c…"
67879,"""Luyện thi N3""",67872,6,"""Nhà Sách Tiki > Sách tiếng Việ…","""https://tiki.vn/luyen-thi-n3/c…"


## Bảng Seller

In [6]:
print(f"Kích thước bảng Seller: {df_seller.shape}")
print(f"Kiểu dữ liệu của bảng Seller:\n{df_seller.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Seller:")
display(df_seller.null_count())
display(df_seller.head())

Kích thước bảng Seller: (4705, 4)
Kiểu dữ liệu của bảng Seller:
[String, String, Float64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Seller:


seller_id,seller_name,seller_rating,total_reviews
u32,u32,u32,u32
0,0,0,0


seller_id,seller_name,seller_rating,total_reviews
str,str,f64,i64
"""sach-tieng-nhat-jlpt""","""SÁCH TIẾNG NHẬT JLPT""",5.0,1
"""noi-that-nha-minh""","""NỘI THẤT ADORA NEMADORA""",4.8,32
"""bookcity-vn""","""BOOKCITY.VN""",4.8,1200
"""espp""","""ESPP""",4.5,117
"""2hbooks""","""2HBooks""",4.9,168


In [7]:
df_seller.unique(subset=["seller_id"], keep="first")\
        .with_columns([
            pl.col("seller_id").str.strip_chars(),
            pl.col("seller_name").str.strip_chars()
        ])\
        .with_columns([
            pl.col("seller_rating").cast(pl.Float32),
            pl.col("total_reviews").cast(pl.Int32)
        ])\
        .with_columns([
            pl.col("seller_name").fill_null("Đang cập nhật"),
            pl.col("seller_rating").fill_null(0.0),
            pl.col("total_reviews").fill_null(0)
        ])

seller_id,seller_name,seller_rating,total_reviews
str,str,f32,i32
"""chan-thien-nano""","""CHÂN THIỆN NANO""",4.7,19
"""hasan""","""HASAN""",4.4,1000
"""viko-mart""","""ViKo Mart""",4.7,545
"""tshop""","""Tshop""",4.1,371
"""kogi-ginseng""","""Kogi Ginseng""",4.7,94
…,…,…,…
"""coffee-trading""","""Coffee Trading""",4.6,21
"""vien-dong-mobile""","""Viễn Đông Mobile""",4.4,194
"""cong-ty-tnhh-vuong-vo""","""CÔNG TY TNHH VƯƠNG VÕ""",5.0,1


## Bảng Product

In [8]:
print(f"Kích thước bảng Product: {df_product.shape}")
print(f"Kiểu dữ liệu của bảng Product:\n{df_product.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Product:")
display(df_product.null_count())
display(df_product.head(10))

Kích thước bảng Product: (310110, 11)
Kiểu dữ liệu của bảng Product:
[String, String, String, String, String, String, String, String, Int64, Int64, Float64]
Kiểm tra số lượng dữ liệu null trong bảng Product:


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,134682,0,17,0,0,2526,0,0,0


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
str,str,str,str,str,str,str,str,i64,i64,f64
"""275822950""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""atzstore""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""GREENCASE""",0,0,0.0
"""273982699""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""phu-kien-bingo""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""ULTRA THIN""",0,0,0.0
"""275236637""","""Luyện Thi Năng Lực Nhật Ngữ Tr…","""N/A""","""67878""","""tien-tho-book-hn""","""https://tiki.vn/luyen-thi-nang…","""https://salt.tikicdn.com/cache…","""""",0,0,0.0
"""273454695""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""N/A""","""67878""","""info-book""","""https://tiki.vn/2500-tu-vung-c…","""https://salt.tikicdn.com/cache…","""ARC ACADEMY""",0,0,0.0
"""273423214""","""Giáo Trình Luyện Thi Năng Lực …","""N/A""","""67878""","""nhan-van""","""https://tiki.vn/giao-trinh-luy…","""https://salt.tikicdn.com/cache…","""NHÓM TÁC GIẢ""",6,0,0.0
"""273377046""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""N/A""","""67878""","""nha-sach-fahasa""","""https://tiki.vn/2500-tu-vung-c…","""https://salt.tikicdn.com/cache…","""ARC ACADEMY""",5,1,5.0
"""271999126""","""500 Câu Hỏi Luyện Thi Năng Lực…","""N/A""","""67878""","""nha-sach-fahasa""","""https://tiki.vn/500-cau-hoi-lu…","""https://salt.tikicdn.com/cache…","""MATSUMOTO, SASAKI HITOKO""",1,0,0.0
"""270785875""","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …","""N/A""","""67878""","""he-thong-nha-sach-abc""","""https://tiki.vn/ky-thi-nang-lu…","""https://salt.tikicdn.com/cache…","""Mua 2 giảm 5K""",0,0,0.0
"""263564208""","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …","""N/A""","""67878""","""he-thong-nha-sach-abc""","""https://tiki.vn/ky-thi-nang-lu…","""https://salt.tikicdn.com/cache…","""Mua 2 giảm 5K""",0,0,0.0


In [9]:
df_product.unique(subset=["product_id"], keep="first")\
            .with_columns([
                pl.col("product_id").cast(pl.Int64),
                pl.col("category_id").cast(pl.Int64),
                pl.col("seller_id").str.strip_chars(),
                
                pl.col("author_brand")
                    .str.replace(r"^0$", "")
                    .replace("", None),
                pl.col("short_description").replace("", None)
            ])\
            .with_columns([
                pl.col("author_brand").fill_null("N/A"),
                pl.col("short_description").fill_null("N/A"),
                pl.col("seller_id").fill_null("N/A")
            ])\
            .with_columns([
                pl.col("sold_quantity").cast(pl.Int32).fill_null(0),
                pl.col("review_count").cast(pl.Int32).fill_null(0),
                pl.col("review_score").cast(pl.Float32).fill_null(0.0)
            ])

product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
i64,str,str,i64,str,str,str,str,i32,i32,f32
276480646,"""Bồi dưỡng học sinh giỏi Sinh h…","""N/A""",2249,"""out_of_stock""","""https://tiki.vn/boi-duong-hoc-…","""https://salt.tikicdn.com/cache…","""N/A""",11,1,5.0
279122741,"""Bộ 3 Đĩa Tròn Sâu Lòng Elmich …","""N/A""",67516,"""joymall-official""","""https://tiki.vn/bo-3-dia-tron-…","""https://salt.tikicdn.com/cache…","""ELMICH""",0,0,0.0
277471948,"""Sách - Giáo Trình Tiếng Nhật T…","""N/A""",67858,"""bamboo-books""","""https://tiki.vn/sach-giao-trin…","""https://salt.tikicdn.com/cache…","""N/A""",0,0,0.0
44057823,"""Đèn Ốp Trần Trang Trí MO5 50w …","""N/A""",23246,"""led-hoang-gia""","""https://tiki.vn/den-op-tran-tr…","""https://salt.tikicdn.com/cache…","""HG""",0,0,0.0
278719557,"""Truyện Tranh - Nisekoi - Cặp Đ…","""N/A""",1084,"""nhan-van""","""https://tiki.vn/truyen-tranh-n…","""https://salt.tikicdn.com/cache…","""NAOSHI KOMI""",4,0,0.0
…,…,…,…,…,…,…,…,…,…,…
273249871,"""Charlie Chaplin Là Ai? _AL""","""N/A""",483,"""abcbooks""","""https://tiki.vn/charlie-chapli…","""https://salt.tikicdn.com/cache…","""PATRICIA BRENNAN DEMUTH, GREGO…",0,0,0.0
276589315,"""Người Thầy - Nguyễn Chí Vịnh (…","""N/A""",6750,"""vinabook-jsc""","""https://tiki.vn/nguoi-thay-tai…","""https://salt.tikicdn.com/cache…","""NGUYỄN CHÍ VỊNH""",110,10,5.0
276103909,"""Nhẫn Akatsuki- Natruto""","""N/A""",18330,"""yapishi-leather""","""https://tiki.vn/nhan-akatsuki-…","""https://salt.tikicdn.com/cache…","""PROLEA PL""",0,0,0.0


In [10]:
def clean_author_brand_advanced(df: pl.DataFrame) -> pl.DataFrame:
    # Tập hợp các từ khóa nhận diện "rác khuyến mãi" (Không phân biệt hoa/thường)
    promo_pattern = r"(?i)(mua\s*\d|giảm\s*\d|giảm\s*k|tặng|freeship|hoàn tiền|voucher|đơn từ)"
    
    return df.with_columns(
        # 1. Cắt khoảng trắng thừa
        pl.col("author_brand").str.strip_chars().alias("author_brand")
    ).with_columns(
        # 2. DÙNG REGEX TÌM RÁC KHUYẾN MÃI: Nếu chứa các từ khóa sale -> Biến thành Null
        pl.when(pl.col("author_brand").str.contains(promo_pattern))
        .then(pl.lit(None))
        .otherwise(pl.col("author_brand"))
        .alias("author_brand")
    ).with_columns(
        # 3. Dọn rác hệ thống thông thường (0, null, na, chuỗi rỗng)
        pl.col("author_brand")
        .str.replace(r"(?i)^(0|null|n/a|na|none)$", "")
        .replace("", None)
        
        # 4. Chuẩn hóa cuối cùng
        .fill_null("N/A")
        .str.to_titlecase()
    )

# Áp dụng hàm
df_product_clean = clean_author_brand_advanced(df_product)

In [11]:
import re

# Làm sạch tên sản phẩm, loại bỏ ký tự rác và cụm quảng cáo phổ biến
def clean_product_name(name):
    if name is None:
        return None

    name = str(name)
    name = name.replace("\xa0", " ")
    name = re.sub(r"\s+", " ", name).strip()

    # bỏ ký tự rác phổ biến
    name = re.sub(r"[|]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    # bỏ cụm quảng cáo thường gặp
    promo_patterns = [
        r"\bchính hãng\b",
        r"\bfreeship\b",
        r"\bgiao nhanh\b",
        r"\bgiá tốt\b",
        r"\bquà tặng\b",
        r"\bkhuyến mãi\b",
        r"\bgiảm\s*\d+%?\b",
    ]

    for pattern in promo_patterns:
        name = re.sub(pattern, "", name, flags=re.IGNORECASE)

    name = re.sub(r"\s+", " ", name).strip(" -–—,.;:")
    return name if name else None

df_product_clean = df_product.with_columns(
    pl.col("product_name")
    .map_elements(clean_product_name, return_dtype=pl.String)
    .alias("product_name_clean")
)

df_product_clean.select([
    "product_id",
    "product_name",
    "product_name_clean"
]).head(10)

product_id,product_name,product_name_clean
str,str,str
"""275822950""","""Ốp lưng dẻo trong dành cho Hua…","""Ốp lưng dẻo trong dành cho Hua…"
"""273982699""","""Ốp lưng dẻo trong dành cho Hua…","""Ốp lưng dẻo trong dành cho Hua…"
"""275236637""","""Luyện Thi Năng Lực Nhật Ngữ Tr…","""Luyện Thi Năng Lực Nhật Ngữ Tr…"
"""273454695""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""2500 Từ Vựng Cần Thiết Cho Kỳ …"
"""273423214""","""Giáo Trình Luyện Thi Năng Lực …","""Giáo Trình Luyện Thi Năng Lực …"
"""273377046""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""2500 Từ Vựng Cần Thiết Cho Kỳ …"
"""271999126""","""500 Câu Hỏi Luyện Thi Năng Lực…","""500 Câu Hỏi Luyện Thi Năng Lực…"
"""270785875""","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …"
"""263564208""","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …","""Kỳ Thi Năng Lực Nhật Ngữ N2 - …"


## Bảng Price Offer

In [12]:
print(f"Kích thước bảng Price_Offer: {df_price_offer.shape}")
print(f"Kiểu dữ liệu của bảng Price_Offer:\n{df_price_offer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:")
display(df_price_offer.null_count())
display(df_price_offer.head())

Kích thước bảng Price_Offer: (949960, 6)
Kiểu dữ liệu của bảng Price_Offer:
[String, String, Float64, Float64, Float64, Datetime(time_unit='us', time_zone=None)]
Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
str,str,f64,f64,f64,datetime[μs]
"""633304a834""","""1217527""",48000.0,48000.0,0.0,2026-02-25 19:20:12.984106
"""7d57439ca7""","""1217917""",80000.0,80000.0,0.0,2026-02-25 19:20:16.744204
"""ca2719fc3b""","""1217919""",80000.0,80000.0,0.0,2026-02-25 19:20:22.292313
"""7a2a3f51a4""","""1217923""",142000.0,142000.0,0.0,2026-02-25 19:20:28.327105
"""52890ddf91""","""1217929""",69900.0,69900.0,0.0,2026-02-25 19:20:31.262106


## Bảng Review

In [13]:
print(f"Kích thước bảng Review: {df_review.shape}")
print(f"Kiểu dữ liệu của bảng Review:\n{df_review.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Review:")
display(df_review.null_count())
display(df_review.head())

Kích thước bảng Review: (1320038, 8)
Kiểu dữ liệu của bảng Review:
[String, String, String, Int64, String, Int64, String, String]
Kiểm tra số lượng dữ liệu null trong bảng Review:


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,165379,0,0,4714


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
str,str,str,i64,str,i64,str,str
"""d2f16143-fe9""","""273377046""","""679284b1e4""",5,"""""",0,"""Đánh giá vào 5 tháng trước""","""Đã dùng 2 ngày"""
"""3105eeae-88b""","""262803182""","""c9a9bab8f4""",4,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 3 tháng"""
"""f0ea86b7-f9b""","""163739509""","""36138811b4""",5,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 4 tháng"""
"""6cbb0f56-49c""","""163739509""","""70b76f17ab""",5,"""sách tốt, như ảnh""",0,"""Đánh giá vào 4 năm trước""","""Đã dùng 3 ngày"""
"""7446cd0a-d0e""","""163739509""","""574ea2859b""",5,"""""",0,"""Đánh giá vào 3 năm trước""","""Đã dùng 4 tháng"""


## Bảng Reviewer

In [14]:
print(f"Kích thước bảng Reviewer: {df_reviewer.shape}")
print(f"Kiểu dữ liệu của bảng Reviewer:\n{df_reviewer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Reviewer:")
display(df_reviewer.null_count())
display(df_reviewer.head())

Kích thước bảng Reviewer: (458141, 5)
Kiểu dữ liệu của bảng Reviewer:
[String, String, String, Int64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Reviewer:


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
u32,u32,u32,u32,u32
0,2,2,0,0


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
str,str,str,i64,i64
"""002304bb1b""","""Đào Quốc Thiện""","""Đã tham gia 7 năm""",21,3
"""ab814822eb""","""Quang Tuệ""","""Đã tham gia 8 năm""",64,0
"""457ec04452""","""Luân Trịnh""","""Đã tham gia 4 năm""",3,0
"""0b4eb78dff""","""Tất Liên""","""Đã tham gia 9 năm""",52,0
"""66407913dc""","""Huệ Nguyễn""","""Đã tham gia 12 năm""",1,0


## Bảng Service

In [15]:
print(f"Kích thước bảng Service: {df_service.shape}")
print(f"Kiểu dữ liệu của bảng Service:\n{df_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Service:")
display(df_service.null_count())
display(df_service.head())

Kích thước bảng Service: (1387, 2)
Kiểu dữ liệu của bảng Service:
[Int64, String]
Kiểm tra số lượng dữ liệu null trong bảng Service:


service_id,service_name
u32,u32
0,0


service_id,service_name
i64,str
249067,"""Trả góp từ 657.500 ₫/tháng"""
249081,"""Trả góp từ 833.250 ₫/tháng"""
249086,"""Trả góp từ 383.250 ₫/tháng"""
249071,"""Trả góp từ 415.833 ₫/tháng"""
249104,"""Trả góp từ 700.833 ₫/tháng"""


## Bảng Coupon

In [16]:
print(f"Kích thước bảng Coupon: {df_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Coupon:\n{df_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Coupon:")
display(df_coupon.null_count())
display(df_coupon.head())

Kích thước bảng Coupon: (1043, 5)
Kiểu dữ liệu của bảng Coupon:
[Int64, String, String, String, Date]
Kiểm tra số lượng dữ liệu null trong bảng Coupon:


coupon_id,coupon_code,title,condition,expiry
u32,u32,u32,u32,u32
0,0,0,0,0


coupon_id,coupon_code,title,condition,expiry
i64,str,str,str,date
4923,"""TET2026""","""Giảm 15K""","""Cho đơn hàng từ 300K""",2026-04-30
1933,"""DEHA53""","""Giảm 50%""","""Cho đơn hàng từ 50K""",2026-02-28
1707,"""10KY26""","""Giảm 10K""","""Cho đơn hàng từ 770K""",2026-02-28
22459,"""PATYCA5""","""Giảm 5K""","""Cho đơn hàng từ 100K""",2026-12-31
259926,"""FEB100KK""","""Giảm 100K""","""Cho đơn hàng từ 3 triệu""",2026-02-28


## Bảng Offer Coupon

In [17]:
print(f"Kích thước bảng Offer_Coupon: {df_offer_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Coupon:\n{df_offer_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:")
display(df_offer_coupon.null_count())
display(df_offer_coupon.head())

Kích thước bảng Offer_Coupon: (1340699, 2)
Kiểu dữ liệu của bảng Offer_Coupon:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:


offer_id,coupon_id
u32,u32
0,0


offer_id,coupon_id
str,i64
"""f4cb55c547""",1
"""f4cb55c547""",2
"""f4cb55c547""",3
"""f4cb55c547""",4
"""f4cb55c547""",5


## Bảng Offer Service

In [18]:
print(f"Kích thước bảng Offer_Service: {df_offer_service.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Service:\n{df_offer_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:")
display(df_offer_service.null_count())
display(df_offer_service.head())

Kích thước bảng Offer_Service: (822582, 2)
Kiểu dữ liệu của bảng Offer_Service:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:


offer_id,service_id
u32,u32
0,0


offer_id,service_id
str,i64
"""282f24cadf""",1
"""282f24cadf""",2
"""f4cb55c547""",1
"""f4cb55c547""",2
"""8c55765cab""",1
